In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("reviewer_raw_fits.csv")

print(df.shape)
print(df.columns.tolist())
df.head()

(1080, 21)
['mu_true', 'phi_true', 'alpha_true', 'gamma_true', 'r_true', 'rep', 'seed', 'model', 'fit_completed', 'all_starts_failed', 'n_starts', 'n_valid_starts', 'runtime_sec', 'mu_est', 'phi_est', 'alpha_est', 'gamma_est', 'r_est', 'selected_start', 'explicit_converged', 'convergence_source']


,mu_true,phi_true,alpha_true,gamma_true,r_true,rep,seed,model,fit_completed,all_starts_failed,...,n_valid_starts,runtime_sec,mu_est,phi_est,alpha_est,gamma_est,r_est,selected_start,explicit_converged,convergence_source
0,-0.11,0.05,-7.7,-2.62,0.004,0,2512176700,joint,True,False,...,2,57.799655,-0.113104,0.001758,-5.496594,-5.000000,0.000417,0,True,success
1,-0.11,0.05,-7.7,-2.62,0.004,0,2512176700,gamma0,True,False,...,2,6.582937,-0.113778,0.317250,-8.054625,0.000000,0.003284,1,False,success
2,-0.11,0.05,-7.7,-2.62,0.004,1,758801068,joint,True,False,...,2,44.689098,-0.111564,-0.091624,-5.622486,-0.422635,0.000365,0,True,success
3,-0.11,0.05,-7.7,-2.62,0.004,1,758801068,gamma0,True,False,...,2,4.949483,-0.112190,-0.792904,-21.860177,0.000000,0.004021,1,False,success
4,-0.11,0.05,-7.7,-2.62,0.004,2,4005173303,joint,True,False,...,2,49.091443,-0.110148,0.013929,-5.732973,4.617812,0.000585,1,True,success


In [3]:
opt_summary = (
    df.groupby("model")
    .agg(
        n_fits=("rep", "size"),
        mean_starts=("n_starts", "mean"),
        min_starts=("n_starts", "min"),
        max_starts=("n_starts", "max"),
        mean_valid_starts=("n_valid_starts", "mean"),
        fit_completion_rate=("fit_completed", "mean"),
        all_starts_failed_rate=("all_starts_failed", "mean"),
        convergence_rate=("explicit_converged", "mean"),
        median_runtime=("runtime_sec", "median"),
    )
    .reset_index()
)

for c in [
    "fit_completion_rate",
    "all_starts_failed_rate",
    "convergence_rate"
]:
    opt_summary[c] *= 100

opt_summary.round(3)

,model,n_fits,mean_starts,min_starts,max_starts,mean_valid_starts,fit_completion_rate,all_starts_failed_rate,convergence_rate,median_runtime
0,gamma0,540,2.0,2,2,2.0,100.0,0.0,38.519,7.639
1,joint,540,2.0,2,2,2.0,100.0,0.0,100.000,51.564


In [4]:
df.groupby("model")["convergence_source"].value_counts(dropna=False)

model   convergence_source
gamma0  success               540
joint   success               540
Name: count, dtype: int64

In [5]:
df.groupby("model")["n_starts"].value_counts().sort_index()

model   n_starts
gamma0  2           540
joint   2           540
Name: count, dtype: int64

In [7]:
df.groupby("model")["selected_start"].value_counts(normalize=True)

model   selected_start
gamma0  0                 0.520370
        1                 0.479630
joint   1                 0.501852
        0                 0.498148
Name: proportion, dtype: float64

In [11]:
df1 = pd.read_csv("reviewer_all_optimizer_starts.csv")

print(df1.shape)
print(df1.columns.tolist())
df1.head()

(2160, 14)
['mu_true', 'phi_true', 'alpha_true', 'gamma_true', 'r_true', 'rep', 'seed', 'model', 'start', 'valid', 'selection_loglik', 'explicit_converged', 'convergence_source', 'error']


,mu_true,phi_true,alpha_true,gamma_true,r_true,rep,seed,model,start,valid,selection_loglik,explicit_converged,convergence_source,error
0,-0.11,0.05,-7.7,-2.62,0.004,0,2512176700,joint,0,True,84.937753,True,success,NaN
1,-0.11,0.05,-7.7,-2.62,0.004,0,2512176700,joint,1,True,84.240057,True,success,NaN
2,-0.11,0.05,-7.7,-2.62,0.004,0,2512176700,gamma0,0,True,84.004516,False,success,NaN
3,-0.11,0.05,-7.7,-2.62,0.004,0,2512176700,gamma0,1,True,84.238033,False,success,NaN
4,-0.11,0.05,-7.7,-2.62,0.004,1,758801068,joint,0,True,81.074357,True,success,NaN


In [15]:
keys = [
    "mu_true", "alpha_true", "gamma_true",
    "phi_true", "r_true", "rep", "model"
]

usable = df1[
    np.isfinite(df1["selection_loglik"])
].copy()

gaps = (
    usable
    .groupby(keys)["selection_loglik"]
    .apply(
        lambda x:
        x.nlargest(2).iloc[0] - x.nlargest(2).iloc[1]
        if len(x) >= 2 else np.nan
    )
    .reset_index(name="best_second_gap")
)

gaps.groupby("model")["best_second_gap"].describe()

,count,mean,std,min,25%,50%,75%,max
model,,,,,,,,
gamma0,540.0,2.510049,29.973608,0.000071,0.109066,0.412046,1.344299,695.473102
joint,540.0,2.340571,21.041337,0.003000,0.336635,0.732013,1.266285,439.091476


In [13]:
keys = [
    "mu_true", "alpha_true", "gamma_true",
    "phi_true", "r_true", "rep", "model"
]

gaps = (
    df1[df1["valid"]]
    .sort_values(keys + ["selection_loglik"], ascending=True)
    .groupby(keys)["selection_loglik"]
    .apply(lambda x: x.nlargest(2).iloc[0] - x.nlargest(2).iloc[1]
           if len(x) >= 2 else np.nan)
    .reset_index(name="best_second_gap")
)

gaps.groupby("model")["best_second_gap"].describe()

,count,mean,std,min,25%,50%,75%,max
model,,,,,,,,
gamma0,540.0,2.510049,29.973608,0.000071,0.109066,0.412046,1.344299,695.473102
joint,540.0,2.340571,21.041337,0.003000,0.336635,0.732013,1.266285,439.091476


In [10]:
keys = [
    "mu_true", "alpha_true", "gamma_true",
    "phi_true", "r_true", "rep", "model"
]

usable = df[
    np.isfinite(df["selection_loglik"])
].copy()

gaps = (
    usable
    .groupby(keys)["selection_loglik"]
    .apply(
        lambda x:
        x.nlargest(2).iloc[0] - x.nlargest(2).iloc[1]
        if len(x) >= 2 else np.nan
    )
    .reset_index(name="best_second_gap")
)

gaps.groupby("model")["best_second_gap"].describe()

NameError: name 'attempts' is not defined

In [16]:
import pandas as pd

mc = pd.read_csv("reviewer_likelihood_mcse.csv")

p500 = mc[mc["particles"] == 500].copy()

raw_nll = p500[
    p500["quantity"].isin(["joint_nll", "gamma0_nll"])
].copy()

raw_nll.head()

,particles,quantity,mean,sd,mcse,repeats,mu_true,alpha_true,gamma_true,phi_true,r_true,rep,seed
0,500,joint_nll,-84.484977,0.664411,0.210105,10,-0.11,-7.7,-2.62,0.05,0.004,0,2512176700
1,500,gamma0_nll,-84.096385,0.137237,0.043398,10,-0.11,-7.7,-2.62,0.05,0.004,0,2512176700
8,500,joint_nll,-80.077020,0.464825,0.146991,10,-0.11,-7.7,-2.62,0.05,0.004,1,758801068
9,500,gamma0_nll,-80.391617,0.000455,0.000144,10,-0.11,-7.7,-2.62,0.05,0.004,1,758801068
16,500,joint_nll,-75.054919,0.864787,0.273470,10,-0.11,-7.7,-2.62,0.05,0.004,2,4005173303


In [17]:
raw_nll_summary = (
    raw_nll
    .groupby("quantity")
    .agg(
        n=("mcse", "size"),
        min_mcse=("mcse", "min"),
        median_mcse=("mcse", "median"),
        mean_mcse=("mcse", "mean"),
        max_mcse=("mcse", "max"),
    )
    .reset_index()
)

raw_nll_summary

,quantity,n,min_mcse,median_mcse,mean_mcse,max_mcse
0,gamma0_nll,540,0.000052,0.116995,0.246076,9.890882
1,joint_nll,540,0.092779,0.222251,0.304327,14.006018


In [18]:
raw_nll_level = (
    raw_nll
    .groupby("quantity")
    .agg(
        median_nll=("mean", "median"),
        mean_nll=("mean", "mean"),
        median_mcse=("mcse", "median"),
    )
    .reset_index()
)

raw_nll_level

,quantity,median_nll,mean_nll,median_mcse
0,gamma0_nll,-37.293869,-20.414458,0.116995
1,joint_nll,-37.843233,-23.215379,0.222251
